# Centering Date Method Tutorial

This tutorial estimates cumulative milk yield with the **centering date method**, also called the McDowell method.

The method treats one test-day yield as representative of its calendar month and multiplies that yield by the number of days assigned to the month. It is intended for data with at most one test per calendar month for each lactation.

In this tutorial we will use the centering date method to calculate total lactation milk production of a self created fake data set and look at the different model properties associated with this method. 

If the lactationcurve package is not installed yet, run:

```bash
% pip install lactationcurve
```

## Step 1: Import the packages used in this tutorial


In [1]:
import pandas as pd

from lactationcurve.characteristics import (
    centering_date_method,
    centering_date_method_single_lactation,
)

c:\Users\Meike van Leerdam\bovi\.venv\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## Step 2: Create monthly test-day data of a single cow

`TestCalendarDate` determines the calendar month. `DaysInMilk` is used for the start and end adjustments, and `MilkingYield` contains the test-day yield.

March is intentionally missing. The method estimates a missing month from the nearest yields before and after it.

In [2]:
records = pd.DataFrame(
    {
        "TestId": [101, 101, 101, 101, 101],
        "TestCalendarDate": pd.to_datetime(
            ["2024-01-10", "2024-02-10", "2024-04-10", "2024-05-10", "2024-06-10"]
        ),
        "DaysInMilk": [10, 41, 101, 131, 162],
        "MilkingYield": [20.0, 30.0, 28.0, 26.0, 24.0],
    }
)

records

,TestId,TestCalendarDate,DaysInMilk,MilkingYield
0,101,2024-01-10,10,20.0
1,101,2024-02-10,41,30.0
2,101,2024-04-10,101,28.0
3,101,2024-05-10,131,26.0
4,101,2024-06-10,162,24.0


## Step 3: Calculate one lactation

Use `centering_date_method_single_lactation` when the dataframe contains one lactation. 

In [6]:
single_result = centering_date_method_single_lactation(records)

print(f"Predicted milk yield over the 305 day lactation period: {single_result} kg")

Predicted milk yield over the 305 day lactation period: 7717.0 kg


## Step 4: Calculate several lactations

Use `centering_date_method` for a dataframe containing multiple `TestId` values. It returns one row per lactation with the 305 milk yield in kg.

In [8]:
second_lactation = records.assign(
    TestId=202,
    MilkingYield=records["MilkingYield"] - 3,
)
all_lactations = pd.concat([records, second_lactation], ignore_index=True)

centering_date_method(all_lactations, lactation_length=305)

,TestId,LactationMilkYield
0,101,7717.0
1,202,6803.5


## Step 5: Use a shorter or longer lactation lenght

Use `lactation_length` to set the desired length of the lactation over which the 305 milk yield must be calculated.

In [10]:
shorter_lactation = centering_date_method_single_lactation(
    records,
    lactation_length=180,
)

longer_lactation = centering_date_method_single_lactation(
    records,
    lactation_length=350,
)

print(f"Predicted milk yield over the 180 day lactation period: {shorter_lactation} kg")
print(f"Predicted milk yield over the 350 day lactation period: {longer_lactation} kg")
print(f"Predicted milk yield over the 305 day lactation period: {single_result} kg")

Predicted milk yield over the 180 day lactation period: 4717.0 kg
Predicted milk yield over the 350 day lactation period: 8797.0 kg
Predicted milk yield over the 305 day lactation period: 7717.0 kg


## Step 6: Exclude an early grace period

Use `lactation_start_grace_period` when early colostrum days should not contribute to the total. The first interval is shortened by the requested number of days.

In [11]:
grace_period_result = centering_date_method_single_lactation(
    records,
    lactation_length=305,
    lactation_start_grace_period=5,
)

print(f"Without grace period: {single_result:.1f} kg")
print(f"With a 5-day grace period: {grace_period_result:.1f} kg")

Without grace period: 7717.0 kg
With a 5-day grace period: 7617.0 kg


## Step 7: Multiple measurements per month

When there are multiple measurements per month, they are averaged together and considered as one measurement per month.



In [13]:
records_with_double_january_test = pd.DataFrame(
    {
        "TestId": [101, 101, 101, 101, 101],
        "TestCalendarDate": pd.to_datetime(
            ["2024-01-10", "2024-01-12", "2024-04-10", "2024-05-10", "2024-06-10"]
        ),
        "DaysInMilk": [10, 41, 101, 131, 162],
        "MilkingYield": [20.0, 30.0, 28.0, 26.0, 24.0],
    }
)

results = centering_date_method_single_lactation(
    records_with_double_january_test,
    lactation_length=305,
)

print(f"Predicted milk yield over the 305 day lactation period: {results} kg")

Predicted milk yield over the 305 day lactation period: 7665.5 kg


## What you learned

- Use `TestCalendarDate` because the method assigns weights by calendar month.
- Use `centering_date_method_single_lactation` for one lactation and `centering_date_method` for multiple lactations.
- Missing months are filled from the nearest available yields.
- The first and last months receive partial-period adjustments.
- Set `lactation_length=305` for the usual 305-day estimate, or provide the actual lactation horizon when it is known.
- Use `lactation_start_grace_period` to exclude early colostrum days.
- If a month has multiple measurements per `TestId`, the measurements will be averaged and considered as one measurement.